**Table of contents**<a id='toc0_'></a>    
- 1. [Read and clean data](#toc1_)    
  - 1.1. [Checking the scope of compromise with balancing](#toc1_1_)    
- 2. [Some descriptive statistics](#toc2_)    
- 3. [Simple Panel Regression](#toc3_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# PWT dataproject

In this project we will explore aggregate data from PWT from national accounts. The project is inspired by Mathias BA project which aims to estimate the elasticity of substitution between capital and labour. The project focuses on simple data manipulation and cleaning techniques along with descriptive statistics and statistical modelling. Importantly the mathematics behind estimation-equations are left out since it is not the focus of the project. 

The data includes homogenous collected, thus comparable, data on aggregate variables, from 1950-2019 for 183 countries. 

In [2]:
# imports
from urllib.request import urlretrieve
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from matplotlib_venn import venn2

# statistical modelling packages
import statsmodels 
from linearmodels import PanelOLS

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# user written modules
import Countries as con # contains country groups, OECD, IMF Advanced economies, IMF Emerging economies

## 1. <a id='toc1_'></a>[Read and clean data](#toc0_)

Below we read the data using an API.

In [20]:
# URL of the stata file
url = "https://dataverse.nl/api/access/datafile/354098"

# Read the stata file into a DataFrame
pwt_csv = pd.read_stata(url)

print(f'There are {len(pwt_csv.countrycode.unique())} unique countries in the dataframe')

There are 183 unique countries in the dataframe


In [3]:
# Import the entire PWT from manual download
pwt_csv = pd.read_csv('data\pwt1001.csv',index_col=0)

For ease of writing and understanding some variables are renamed. The variables hc is a human capital index, and labsh is the labor share of compensation. The dataframe is filtered to remove all unessecary data. 

In [22]:
# rename variables to well-known economic variables
pwt_csv = pwt_csv.rename(columns = {'rgdpna':'Y', "rnna":"K", "emp": "L"})

# filter the df for the variables of interest
var_of_int = ["countrycode", "year", "Y", "K", "L", "hc", "labsh"]

# remove all variables not in var_of_int
pwt_csv = pwt_csv[var_of_int]

We now clean the data from missing values, though there are few at this point since missing data is simply omitted from the set from PWT's side. We also generate some new variables which are of interest. 

In [23]:
# drop nan and reset index
pwt_csv.dropna(inplace=True)
pwt_csv.reset_index(inplace=True, drop=True) 

# Generate variables of interest
pwt_csv["cap_out"] = pwt_csv["K"] / pwt_csv["Y"] # capital output ratio
pwt_csv["HC"] = pwt_csv["hc"] * pwt_csv["L"] # aggregate human capital
pwt_csv["capsh"] = 1 - pwt_csv["labsh"] # capital share as reciproc of labsh

# log transform
pwt_csv["lnY"] = np.log(pwt_csv["Y"])
pwt_csv["lnK"] = np.log(pwt_csv["K"])
pwt_csv["ln_capsh"] = np.log(pwt_csv["capsh"])
pwt_csv["ln_labsh"] = np.log(pwt_csv["labsh"])
pwt_csv["ln_cap_out"] = np.log(pwt_csv["K"] / pwt_csv["Y"])
pwt_csv["ln_cap_out_neg"] = -np.log(pwt_csv["K"] / pwt_csv["Y"])

# Sort values by countrycode and year to make functions possible
pwt_csv.sort_values(by=["countrycode", "year"])

,countrycode,year,Y,K,L,hc,labsh,cap_out,HC,capsh,lnY,lnK,ln_capsh,ln_labsh,ln_cap_out,ln_cap_out_neg
0,AGO,1970,54237.055,257238.100,3.666207,1.015686,0.284385,4.742848,3.723715,0.715615,10.901120,12.457757,-0.334613,-1.257427,1.556638,-1.556638
1,AGO,1971,57491.277,274282.500,3.742484,1.018196,0.284385,4.770854,3.810581,0.715615,10.959389,12.521914,-0.334613,-1.257427,1.562525,-1.562525
2,AGO,1972,57606.260,290921.900,3.853271,1.020711,0.284385,5.050179,3.933078,0.715615,10.961387,12.580810,-0.334613,-1.257427,1.619424,-1.619424
3,AGO,1973,62272.367,309321.280,3.987807,1.023234,0.284385,4.967232,4.080459,0.715615,11.039273,12.642136,-0.334613,-1.257427,1.602863,-1.602863
4,AGO,1974,64202.810,328087.000,4.130696,1.025762,0.284385,5.110166,4.237112,0.715615,11.069802,12.701034,-0.334613,-1.257427,1.631232,-1.631232
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6809,ZWE,2015,42008.200,53550.227,6.393752,2.584653,0.533381,1.274757,16.525630,0.466619,10.645620,10.888375,-0.762242,-0.628519,0.242755,-0.242755
6810,ZWE,2016,42325.727,55167.008,6.504374,2.616257,0.533381,1.303392,17.017116,0.466619,10.653150,10.918120,-0.762242,-0.628519,0.264970,-0.264970
6811,ZWE,2017,44316.742,56829.140,6.611773,2.648248,0.533381,1.282340,17.509615,0.466619,10.699118,10.947805,-0.762242,-0.628519,0.248687,-0.248687
6812,ZWE,2018,46457.098,58552.810,6.714952,2.680630,0.533381,1.260363,18.000300,0.466619,10.746285,10.977684,-0.762242,-0.628519,0.231400,-0.231400


## Subsetting and balancing panel data 

Untill know we have worked with the entire country sample from PWT, we know generate new datasets containing only countries in certain international economic institutions, OECD, IMF. These groups are specified in the countries.py file.  

In [24]:
# List of the 4 groups of interest
# entire set
world_data = pwt_csv.copy()

# Define new dataframes as the country groups in the py-file
oecd_data = pwt_csv[pwt_csv['countrycode'].isin(con.oecd_countries)]
imf_advanced_data = pwt_csv[pwt_csv['countrycode'].isin(con.imf_advanced_countries)]
imf_emerging_data = pwt_csv[pwt_csv['countrycode'].isin(con.imf_emerging_countries)]

When working with panel data it is important to be aware of the dataset structure. We refer to a panel as being either balanced or unbalanced. In regression, this is important since it can require different methods and yield different results. To simplify, we make the entire panel balanced from 1960 (could possible find a nice way to determine a balance between amount of countries and years?)

In [25]:
# Make them balanced  
# Group by country code and find the minimum year for each country
min_years_by_country = pwt_csv.groupby('countrycode')['year'].min()

# Decide that 1960 is the latest allowed starting point
# Delete all countries where starting year is above 1960 

# Filter out countries where the starting year is after 1960
countries_before_1960 = min_years_by_country[min_years_by_country <= 1960].index

# Filter the DataFrame to include only countries with starting year before or in 1960
balanced_data = pwt_csv[pwt_csv['countrycode'].isin(countries_before_1960)]

# Filter out all observations before 1960 
balanced_data = balanced_data[balanced_data['year'] > 1959]

# Filtering the data into the 4 groups 
world_bal_data = balanced_data.copy()
oecd_bal_data = balanced_data[balanced_data['countrycode'].isin(con.oecd_countries)]
imf_advanced_bal_data = balanced_data[balanced_data['countrycode'].isin(con.imf_advanced_countries)]
imf_emerging_bal_data = balanced_data[balanced_data['countrycode'].isin(con.imf_emerging_countries)]


To check whether the restriction in using 1960 as the cutoff year for balancing was reasonable, we are interested in seing how many countries were omitted. 

In [30]:
# How many countries are removed 
num_countries_before = len(pwt_csv['countrycode'].unique())
num_countries_after = len(balanced_data['countrycode'].unique())

# Define functions to calculate the number of countries in each group
def num_countries_in_group(group_name, country_list):
    return len(pwt_csv[pwt_csv['countrycode'].isin(country_list)]['countrycode'].unique())

def num_countries_in_group_after_filter(group_name, country_list):
    return len(balanced_data[balanced_data['countrycode'].isin(country_list)]['countrycode'].unique())

# Calculate the number of countries in each group before filtering
num_countries_in_groups_before = {
    "world": num_countries_in_group("world", pwt_csv['countrycode'].unique()),
    "oecd": num_countries_in_group("oecd", con.oecd_countries),
    "imf_advanced": num_countries_in_group("imf_advanced", con.imf_advanced_countries),
    "imf_emerging": num_countries_in_group("imf_emerging", con.imf_emerging_countries)
}

# Calculate the number of countries in each group after filtering
num_countries_in_groups_after = {
    "world": num_countries_in_group_after_filter("world", pwt_csv['countrycode'].unique()),
    "oecd": num_countries_in_group_after_filter("oecd", con.oecd_countries),
    "imf_advanced": num_countries_in_group_after_filter("imf_advanced", con.imf_advanced_countries),
    "imf_emerging": num_countries_in_group_after_filter("imf_emerging", con.imf_emerging_countries)
}

# Calculate the number of countries removed from each group
num_countries_removed_from_groups = {
    group: num_countries_in_groups_before[group] - num_countries_in_groups_after[group]
    for group in num_countries_in_groups_before
}

# Print the results
for group, num_removed in num_countries_removed_from_groups.items():
    print(f"Number of countries removed from {group} group: {num_removed}")

Number of countries removed from world group: 40
Number of countries removed from oecd group: 8
Number of countries removed from imf_advanced group: 8
Number of countries removed from imf_emerging group: 32


## 2. <a id='toc2_'></a>[Some descriptive statistics](#toc0_)

Could do some interactive plots on capital output relationship or capital labor shares, differences between countries or group averages. 

In [ ]:
pwt_csv['g_K'] = pwt_csv['K'].pct_change(periods=1)*100
pwt_csv['g_Y'] = pwt_csv['Y'].pct_change(periods=1)*100

In [ ]:
pwt_csv_grouped = pwt_csv.groupby(by=["countrycode"])

## 3. <a id='toc3_'></a>[Simple Panel Regression](#toc0_)


Estimating the EOS between capital and labour, $\sigma$, defined as $\frac{1}{1+\phi}$, from the following equation $\ln(\frac{rK}{Y})_{it} = c_i+d_t-\phi\ln(\frac{K}{Y})_{it}+\epsilon_{it}$, which follows from a CES production function, using the first order condition an algebraic manipulation. These variable are naturally stationary, such that the FE estimator is consistent and unbiased. 

We include fixed effects to capture the unobserved characteristics of countries, any factors that are constant over time for each individual but different across countries. This could be differences in institutions, culture, natural ressources. 

We include fixed to similarly capture the unobserved factors that affect all countries in the panel similarly across time periods. This could be macroeconomic conditions such as global economic cycles, technological changes or policy shifts.  

In [ ]:
def fixed_effects(data, fixed_effects, time_effects):
    data = data.set_index(["countrycode", "year"])
    a = data["ln_capsh"]
    cap_out = data["ln_cap_out_neg"]
    
    res = PanelOLS(a, cap_out, entity_effects=fixed_effects, time_effects=time_effects).fit(cov_type="clustered", cluster_entity=True)
    sigma = 1/(1+res.params)
    return sigma

print("oecd results")
print(fixed_effects(oecd_data, True, True))
